In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import pickle

# 1. DATA LOADING & PREPARATION
print("Loading data...")
# Read the CSV files and assign labels (0 for Fake, 1 for True)
fake_df = pd.read_csv('Fake.csv')
fake_df['label'] = 0 

true_df = pd.read_csv('True.csv')
true_df['label'] = 1

# Combine and shuffle the dataset
data = pd.concat([fake_df, true_df], axis=0, ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features and labels
X = data['text']
y = data['label']

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# 2. BUILDING THE 5-MODEL ENSEMBLE
print("Building the 5-Model Ensemble...")

# Define the individual "voter" models
model_1 = LogisticRegression(max_iter=1000)
model_2 = RandomForestClassifier(n_estimators=100, n_jobs=-1)
model_3 = SVC(probability=True, kernel='linear')
model_4 = MultinomialNB()
model_5 = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1)

# Combine them into a Soft Voting Classifier
ensemble_classifier = VotingClassifier(
    estimators=[
        ('lr', model_1), 
        ('rf', model_2), 
        ('svc', model_3),
        ('nb', model_4),
        ('gb', model_5)
    ],
    voting='soft'
)

# Pipeline linking TF-IDF and the Ensemble
clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_df=0.7)),
    ('ensemble', ensemble_classifier)
])

# 3. TRAINING AND EVALUATING
print("Training the ensemble (This will take a few minutes due to multiple models)...")
clf_pipeline.fit(X_train, y_train)

print("Evaluating the model...")
y_pred = clf_pipeline.predict(X_test)

# Print the performance metrics
print(classification_report(y_test, y_pred))

# 4. SAVING PICKLE FILE FOR THE WEB APP
print("Saving the pipeline...")
with open('fake_news_ensemble_model.pkl', 'wb') as file:
    pickle.dump(clf_pipeline, file)

print("Success! Model saved as 'fake_news_ensemble_model.pkl'.")

Loading data...
Building the 5-Model Ensemble...
Training the ensemble (This will take a few minutes due to multiple models)...
Evaluating the model...
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      4696
           1       0.99      1.00      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Saving the pipeline...
Success! Model saved as 'fake_news_ensemble_model.pkl'.
